### **Create DIM_CHANNEL in Duckdb 🦆**

In [1]:
import duckdb

In [2]:
con = duckdb.connect("EXP1.duckdb")

In [3]:
con.execute("""CREATE SEQUENCE channel_seq START 1""")

In [4]:
con.execute("""
CREATE TABLE DIM_CHANNEL (
    channel_sk INTEGER PRIMARY KEY DEFAULT nextval('channel_seq'),
    channel_id INTEGER UNIQUE,
    channel_name VARCHAR
)
""")

In [5]:
con.execute("""CREATE INDEX idx_dim_channel_id ON DIM_CHANNEL(channel_id)""")

In [6]:
con.execute("SELECT * FROM DIM_CHANNEL").fetchall()

[]

In [7]:
con.execute("INSERT INTO DIM_CHANNEL (channel_id, channel_name) VALUES (9, 'tv')")

In [8]:
con.execute("SELECT * FROM DIM_CHANNEL").fetchall()

[(1, 9, 'tv')]

### **Extract the channels.csv batch file 🌞**

In [9]:
channels_url = 'https://raw.githubusercontent.com/AhmedWaheedM/fastfeast-data-solution/refs/heads/salma-chaos/batch/channels.csv'

In [10]:
con.sql(f"""
CREATE TEMP TABLE temp_DIM_CHANNEL AS
SELECT *
FROM read_csv_auto('{channels_url}', all_varchar=true)
""")

### **Validate the channels.csv column names🙈**

In [11]:
def validate_columns(url, expected):
    cols = con.sql(f"""
        SELECT *
        FROM '{url}'
        LIMIT 0
    """).columns

    if cols.sort() != expected.sort():
        return False

    return True

In [12]:
validate_columns(channels_url, ['channel_name', 'channel_ id'])

True

### **Validate the channels.csv columns data types 🙂‍↔️**

In [13]:
invalid_data_types_df = con.sql("""
SELECT *
FROM temp_DIM_CHANNEL
WHERE TRY_CAST(channel_id   AS INTEGER) IS NULL
OR    TRY_CAST(channel_name AS VARCHAR) IS NULL
""")

In [14]:
valid_data_types_df = con.sql("""
SELECT *
FROM temp_DIM_CHANNEL
WHERE TRY_CAST(channel_id   AS INTEGER) IS NOT NULL
OR    TRY_CAST(channel_name AS VARCHAR) IS NOT NULL
""")

### **Validate the channels.csv columns nullabilty 🫗**

In [15]:
invalid_nullables_df = valid_data_types_df.filter("""
   channel_id    IS NULL
OR channel_name  IS NULL
""")

In [16]:
valid_nullables_df = valid_data_types_df.filter("""
   channel_id    IS NOT NULL
OR channel_name  IS NOT NULL
""")

### **Convert valid channels.csv rows into actual data types 💒**

In [17]:
con.register("valid_nullables_df", valid_nullables_df)

In [18]:
proper_typed_df = con.sql("""
SELECT
    CAST(channel_id AS INTEGER) AS channel_id,
    channel_name
FROM valid_nullables_df
""")

### **Validate the channels.csv primary key uniqueness 🦹‍♀️**

In [19]:
con.register("proper_typed_df", proper_typed_df)

In [20]:
id_violation_df = con.sql("""
SELECT df.channel_id, df.channel_name
FROM DIM_CHANNEL dim
JOIN proper_typed_df df
ON dim.channel_id = df.channel_id
""")

In [21]:
con.execute("""
INSERT INTO DIM_CHANNEL (channel_id , channel_name)
SELECT df.channel_id, df.channel_name
FROM DIM_CHANNEL dim
JOIN proper_typed_df df
ON dim.channel_id != df.channel_id
""")

In [22]:
con.sql("SELECT * FROM DIM_CHANNEL")

┌────────────┬────────────┬──────────────┐
│ channel_sk │ channel_id │ channel_name │
│   int32    │   int32    │   varchar    │
├────────────┼────────────┼──────────────┤
│          1 │          9 │ tv           │
│          2 │          1 │ app          │
│          3 │          2 │ chat         │
│          4 │          3 │ phone        │
│          5 │          4 │ email        │
└────────────┴────────────┴──────────────┘

### **Validate the channels.csv refrential integrity 🌚**